In [124]:
import os
from typing import List, Dict, Any
import pandas as pd
from rich import print

In [4]:
from langchain_core.documents import Document
from langchain_text_splitters import(
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
print("Set up Completed!")

Set up Completed!


In [5]:
## create a simple document
doc=Document(
    page_content="This is the main text content that will be embedded and searched.",
    metadata={
        "source":"example.txt",
        "page":1,
        "author":"Abhijeet B",
        "date_created":"2024-01-01",
        "cutom_field":"any_value"
    }
)
print("Document Structure")

print(f"Content :{doc.page_content}")
print(f"Metadata :{doc.metadata}")

# Why metadata matters:
print("\n📝 Metadata is crucial for:")
print("- Filtering search results")
print("- Tracking document sources")
print("- Providing context in responses")
print("- Debugging and auditing")

Document Structure
Content :This is the main text content that will be embedded and searched.
Metadata :{'source': 'example.txt', 'page': 1, 'author': 'Abhijeet B', 'date_created': '2024-01-01', 'cutom_field': 'any_value'}

📝 Metadata is crucial for:
- Filtering search results
- Tracking document sources
- Providing context in responses
- Debugging and auditing


In [6]:
type(doc)

langchain_core.documents.base.Document

In [7]:
## Create a simple txt file
import os
os.makedirs("data/text_files",exist_ok=True)

In [8]:
sample_texts={
    "data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [9]:
from langchain_community.document_loaders import TextLoader

## Loading a single text file
loader=TextLoader("data/text_files/python_intro.txt", encoding="utf-8")

documents=loader.load()
print(f"📄 Loaded {len(documents)} document")
print(f"Content preview: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")


📄 Loaded 1 document
Content preview: Python Programming Introduction

Python is a high-level, interpreted programming language known for ...
Metadata: {'source': 'data/text_files/python_intro.txt'}


In [10]:
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=True

)

documents=dir_loader.load()

print(f"📁 Loaded {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"\nDocument {i+1}:")
    print(f"  Source: {doc.metadata['source']}")
    print(f"  Length: {len(doc.page_content)} characters")


# 📊 Analysis
print("\n📊 DirectoryLoader Characteristics:")
print("✅ Advantages:")
print("  - Loads multiple files at once")
print("  - Supports glob patterns")
print("  - Progress tracking")
print("  - Recursive directory scanning")

print("\n❌ Disadvantages:")
print("  - All files must be same type")
print("  - Limited error handling per file")
print("  - Can be memory intensive for large directories")

100%|██████████| 2/2 [00:00<00:00, 1099.86it/s]

📁 Loaded 2 documents

Document 1:
  Source: data\text_files\machine_learning.txt
  Length: 573 characters

Document 2:
  Source: data\text_files\python_intro.txt
  Length: 489 characters

📊 DirectoryLoader Characteristics:
✅ Advantages:
  - Loads multiple files at once
  - Supports glob patterns
  - Progress tracking
  - Recursive directory scanning

❌ Disadvantages:
  - All files must be same type
  - Limited error handling per file
  - Can be memory intensive for large directories


### Text Splitting Statergies

In [11]:
# Method 1: Character-based splitting
text=documents[0].page_content
print("1️⃣ CHARACTER TEXT SPLITTER")
char_splitter = CharacterTextSplitter(
    separator=" ",  # Split on newlines
    chunk_size=200,  # Max chunk size in characters
    chunk_overlap=20,  # Overlap between chunks
    length_function=len  # How to measure chunk size
)

char_chunks=char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

1️⃣ CHARACTER TEXT SPLITTER
Created 3 chunks
First chunk: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...


In [12]:
print(char_chunks[0])
print("------------------")
print(char_chunks[1])

Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing
------------------
on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning:


In [13]:
# Method 2: Recursive character splitting (RECOMMENDED)
print("\n2️⃣ RECURSIVE CHARACTER TEXT SPLITTER")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=[" "],  # Try these separators in order
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

recursive_chunks = recursive_splitter.split_text(text)
print(f"Created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0][:100]}...")


2️⃣ RECURSIVE CHARACTER TEXT SPLITTER
Created 4 chunks
First chunk: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...


In [14]:
print(char_chunks[0])
print("------------------")
print(char_chunks[1])

Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing
------------------
on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning:


In [15]:
# Method 3: Token-based splitting
print("\n3️⃣ TOKEN TEXT SPLITTER")
token_splitter = TokenTextSplitter(
    chunk_size=50,  # Size in tokens (not characters)
    chunk_overlap=10
)

token_chunks = token_splitter.split_text(text)
print(f"Created {len(token_chunks)} chunks")
print(f"First chunk: {token_chunks[0][:100]}...")


3️⃣ TOKEN TEXT SPLITTER
Created 3 chunks
First chunk: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...


## Load Pdf Files

In [16]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader
)

In [40]:
### PypdfLoader
print("PyPdfloader")

try:
    pypdf_loader=PyPDFLoader("data/pdf/attention.pdf")
    pypdf_docs=pypdf_loader.load()
    print(f"  Loaded {len(pypdf_docs)} pages")
    print(f"  Page 1 content: {pypdf_docs[0].page_content[:100]}...")
    print(f"  Metadata: {pypdf_docs[0].metadata}")

except Exception as e:
    print(f"Error : {e}")

PyPdfloader
  Loaded 15 pages
  Page 1 content: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and...
  Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/pdf/attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


In [41]:
# Method 2: PyMuPDFLoader (Fast and accurate)
print("\n3️⃣ PyMuPDFLoader")
try:
    pymupdf_loader = PyMuPDFLoader("data/pdf/attention.pdf")
    pymupdf_docs = pymupdf_loader.load()
    
    print(f"  Loaded {len(pymupdf_docs)} pages")
    print(f"  Includes detailed metadata")
except Exception as e:
    print(f"  Error: {e}")


3️⃣ PyMuPDFLoader
  Loaded 15 pages
  Includes detailed metadata


In [19]:
# 📊 PDF Loader Comparison
print("\n📊 PDF Loader Comparison:")
print("\nPyPDFLoader:")
print("  ✅ Simple and reliable")
print("  ✅ Good for most PDFs")
print("  ✅ Preserves page numbers")
print("  ❌ Basic text extraction")
print("  Use when: Standard text PDFs")

print("\nPyMuPDFLoader:")
print("  ✅ Fast processing")
print("  ✅ Good text extraction")
print("  ✅ Image extraction support")
print("  Use when: Speed is important")


📊 PDF Loader Comparison:

PyPDFLoader:
  ✅ Simple and reliable
  ✅ Good for most PDFs
  ✅ Preserves page numbers
  ❌ Basic text extraction
  Use when: Standard text PDFs

PyMuPDFLoader:
  ✅ Fast processing
  ✅ Good text extraction
  ✅ Image extraction support
  Use when: Speed is important


In [20]:
# Example of raw PDF extraction
raw_pdf_text = """Company Financial Report


    The ﬁnancial performance for ﬁscal year 2024
    shows signiﬁcant growth in proﬁtability.
    
    
    
    Revenue increased by 25%.
    
The company's efﬁciency improved due to workﬂow
optimization.


Page 1 of 10
"""

# Apply the cleaning function
def clean_text(text):
    # Remove excessive whitespace
    text = " ".join(text.split())
    
    # Fix ligatures
    text = text.replace("ﬁ", "fi")
    text = text.replace("ﬂ", "fl")
    
    return text

cleaned = clean_text(raw_pdf_text)
print("BEFORE:")
print(repr(raw_pdf_text))
print("\nAFTER:")
print(repr(cleaned))

# Output:
# BEFORE:
# 'Company Financial Report\n\n\n    The ﬁnancial performance for ﬁscal year 2024\n    shows signiﬁcan'
# 
# AFTER:
# 'Company Financial Report The financial performance for fiscal year 2024 shows significant growth in'

BEFORE:
"Company Financial Report\n\n\n    The ﬁnancial performance for ﬁscal year 2024\n    shows signiﬁcant growth in proﬁtability.\n\n\n\n    Revenue increased by 25%.\n\nThe company's efﬁciency improved due to workﬂow\noptimization.\n\n\nPage 1 of 10\n"

AFTER:
"Company Financial Report The financial performance for fiscal year 2024 shows significant growth in profitability. Revenue increased by 25%. The company's efficiency improved due to workflow optimization. Page 1 of 10"


In [21]:
class SmartPDFProcessor:
    """Advanced PDF processing with error handling"""
    def __init__(self,chunk_size=1000,chunk_overlap=100):
        self.chunk_size=chunk_size,
        self.chunk_overlap=chunk_overlap,
        self.text_splitter=RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=[" "],
        )

    def process_pdf(self,pdf_path:str)->List[Document]:
        """Process PDF with smart chunking and metadata enhancement"""

        # Load PDF
        loader=PyPDFLoader(pdf_path)
        pages=loader.load()

        ## Process each page
        processed_chunks=[]

        for page_num,page in enumerate(pages):
            ## clean text
            cleaned_text=self._clean_text(page.page_content)

            # Skip nearly empty pages
            if len(cleaned_text.strip()) < 50:
                continue

            # Create chunks with enhanced metadata
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text)
                }]
            )
            processed_chunks.extend(chunks)

        return processed_chunks

    def _clean_text(self, text: str) -> str:
        """Clean extracted text"""
        # Remove excessive whitespace
        text = " ".join(text.split())
        
        # Fix common PDF extraction issues
        text = text.replace("ﬁ", "fi")
        text = text.replace("ﬂ", "fl")
        
        return text

In [22]:
preprocessor=SmartPDFProcessor()

In [23]:
## Process a PDF if available
try:
    smart_chunks=preprocessor.process_pdf("data/pdf/attention.pdf")
    print(f"Processed into {len(smart_chunks)} smart chunks")

    # Show enhanced metadata
    if smart_chunks:
        print("\nSample chunk metadata:")
        for key, value in smart_chunks[0].metadata.items():
            print(f"  {key}: {value}")

except Exception as e:
    print(f"Processing error: {e}")

Processed into 49 smart chunks

Sample chunk metadata:
  producer: pdfTeX-1.40.25
  creator: LaTeX with hyperref
  creationdate: 2024-04-10T21:11:43+00:00
  author: 
  keywords: 
  moddate: 2024-04-10T21:11:43+00:00
  ptex.fullbanner: This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5
  subject: 
  title: 
  trapped: /False
  source: data/pdf/attention.pdf
  total_pages: 15
  page: 1
  page_label: 1
  chunk_method: smart_pdf_processor
  char_count: 2858


### What Are Embeddings?
Think of embeddings as a way to translate words into a language that computers understand - numbers!

In [24]:
import numpy as np
import matplotlib.pyplot as plt


In [25]:
def cosine_similarity(vec1, vec2):
    """
    Cosine similarity measures the angle between two vectors.
    - Result close to 1: Very similar
    - Result close to 0: Not related
    - Result close to -1: Opposite meanings
    """

    dot_product=np.dot(vec1,vec2)
    norm_a=np.linalg.norm(vec1)
    norm_b=np.linalg.norm(vec2)
    return dot_product/(norm_a * norm_b)

In [ ]:
### Huggingface And OpenAI Models

from langchain_huggingface import HuggingFaceEmbeddings

## Initialize a simple Embedding model(no API Key needed!)
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

In [42]:
## create your first embeddings
text_1="Cats are amazing creatures"
embedding_1=embeddings.embed_query(text_1)
print(f"Text: {text_1}")
print(f"Embedding length : {len(embedding_1)}")
print(embedding_1[0:5])

text_2="Kittens are amazing creatures"
embedding_2=embeddings.embed_query(text_2)
print(f"Text: {text_2}")
print(f"Embedding length : {len(embedding_2)}")
print(embedding_2[0:5])

text_3="Cars are amazing inventions"
embedding_3=embeddings.embed_query(text_3)
print(f"Text: {text_3}")
print(f"Embedding length : {len(embedding_3)}")
print(embedding_3[0:5])


Text: Cats are amazing creatures
Embedding length : 384
[0.0378708615899086, -0.017460310831665993, 0.06494452804327011, 0.03330286964774132, -0.10574080049991608]
Text: Kittens are amazing creatures
Embedding length : 384
[-0.00813260581344366, -0.01686469092965126, 0.04475962370634079, 0.029233518987894058, -0.06804756075143814]
Text: Cars are amazing inventions
Embedding length : 384
[-0.08040277659893036, 0.07303745299577713, 0.05348362401127815, -0.032346077263355255, -0.04458705335855484]


In [43]:
print(f" Cat and Kitten : {cosine_similarity(embedding_1,embedding_2)}")
print(f" Cat and Car : {cosine_similarity(embedding_1,embedding_3)}")

 Cat and Kitten : 0.8539930320442817
 Cat and Car : 0.3829248334838192


### Open AI Embeddings

In [44]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [45]:
api_key = os.getenv("OPEN_API_KEY")
print(api_key[:5])

sk-pr


In [58]:
from langchain_openai import OpenAIEmbeddings
embeddings_sm=OpenAIEmbeddings(model="text-embedding-3-small", api_key = api_key)
embeddings_lg=OpenAIEmbeddings(model="text-embedding-3-large", api_key = api_key)

In [59]:
## Single text embeddings
single_text="Langchain and Rag are amazing frameworks and projects to work on"
single_embeddings=embeddings_sm.embed_query(single_text)

In [60]:
print("📝 Single Text Embedding:")
print(f"Input: {single_text}")
print(f"Output: Vector of {len(single_embeddings)} dimensions")
print(f"Sample values: {single_embeddings[:5]}")

📝 Single Text Embedding:
Input: Langchain and Rag are amazing frameworks and projects to work on
Output: Vector of 1536 dimensions
Sample values: [-0.05010986328125, -0.0310821533203125, -0.0034637451171875, -0.0032482147216796875, 0.03265380859375]


In [63]:
# Example 2: Multiple texts at once
multiple_texts = [
    "Python is a programming language",
    "LangChain is a framework for LLM applications",
    "Embeddings convert text to numbers",
    "Vectors can be compared for similarity"
]
multiple_embeddings = embeddings_sm.embed_documents(multiple_texts)

In [64]:
print("\n📚 Multiple Text Embeddings:")
print(f"Number of texts: {len(multiple_texts)}")
print(f"Number of embeddings: {len(multiple_embeddings)}")
print(f"Each embedding size: {len(multiple_embeddings[0])}")


📚 Multiple Text Embeddings:
Number of texts: 4
Number of embeddings: 4
Each embedding size: 1536


### Cosine Similarity With OpenAI Embeddings

In [65]:
# Example 1: Finding similar sentences
sentences = [
    "The cat sat on the mat",
    "A feline rested on the rug",
    "The dog played in the yard",
    "I love programming in Python",
    "Python is my favorite programming language"
]

In [66]:
sentence_embeddings = embeddings_sm.embed_documents(sentences)

In [67]:
## Calculate the simialrity betwween all pairs

for i in range(len(sentences)):
    for j in range(i+1,len(sentences)):
        similarity=cosine_similarity(sentence_embeddings[i],sentence_embeddings[j])

        print(f"'{sentences[i]}' vs '{sentences[j]}'")
        print(f"Similarity: {similarity:.3f}\n")


'The cat sat on the mat' vs 'A feline rested on the rug'
Similarity: 0.656

'The cat sat on the mat' vs 'The dog played in the yard'
Similarity: 0.324

'The cat sat on the mat' vs 'I love programming in Python'
Similarity: 0.089

'The cat sat on the mat' vs 'Python is my favorite programming language'
Similarity: 0.120

'A feline rested on the rug' vs 'The dog played in the yard'
Similarity: 0.296

'A feline rested on the rug' vs 'I love programming in Python'
Similarity: 0.055

'A feline rested on the rug' vs 'Python is my favorite programming language'
Similarity: 0.103

'The dog played in the yard' vs 'I love programming in Python'
Similarity: 0.126

'The dog played in the yard' vs 'Python is my favorite programming language'
Similarity: 0.085

'I love programming in Python' vs 'Python is my favorite programming language'
Similarity: 0.708



In [68]:
sentence_embeddings = embeddings_lg.embed_documents(sentences)

In [71]:
print("\n📚 Sentences Text Embeddings using the large model:")
print(f"Number of texts: {len(sentences)}")
print(f"Number of embeddings: {len(sentence_embeddings)}")
print(f"Each embedding size: {len(sentence_embeddings[0])}")


📚 Sentences Text Embeddings using the large model:
Number of texts: 5
Number of embeddings: 5
Each embedding size: 3072


In [69]:
## Calculate the simialrity betwween all pairs

for i in range(len(sentences)):
    for j in range(i+1,len(sentences)):
        similarity=cosine_similarity(sentence_embeddings[i],sentence_embeddings[j])

        print(f"'{sentences[i]}' vs '{sentences[j]}'")
        print(f"Similarity: {similarity:.3f}\n")

'The cat sat on the mat' vs 'A feline rested on the rug'
Similarity: 0.530

'The cat sat on the mat' vs 'The dog played in the yard'
Similarity: 0.334

'The cat sat on the mat' vs 'I love programming in Python'
Similarity: 0.110

'The cat sat on the mat' vs 'Python is my favorite programming language'
Similarity: 0.151

'A feline rested on the rug' vs 'The dog played in the yard'
Similarity: 0.336

'A feline rested on the rug' vs 'I love programming in Python'
Similarity: 0.066

'A feline rested on the rug' vs 'Python is my favorite programming language'
Similarity: 0.093

'The dog played in the yard' vs 'I love programming in Python'
Similarity: 0.143

'The dog played in the yard' vs 'Python is my favorite programming language'
Similarity: 0.131

'I love programming in Python' vs 'Python is my favorite programming language'
Similarity: 0.821



In [72]:
### Example- Semantic Search - Retireve the similar sentence
# Test semantic search
documents = [
    "LangChain is a framework for developing applications powered by language models",
    "Python is a high-level programming language",
    "Machine learning is a subset of artificial intelligence",
    "Embeddings convert text into numerical vectors",
    "The weather today is sunny and warm"
]
query="What is Langchain?"


In [73]:
def semantic_search(query,documents,embeddings_models,top_k=3):
    """Simple semantic search implementation"""

    ## embed query and doument

    query_embedding=embeddings_models.embed_query(query)
    doc_embeddings = embeddings_models.embed_documents(documents)

    ## Calculate the similarity score

    similarties=[]

    for i,doc_emb in enumerate(doc_embeddings):
        similarity=cosine_similarity(query_embedding,doc_emb)
        similarties.append((similarity,documents[i]))

    ## Sort by similarity
    similarties.sort(reverse=True)
    return similarties[:top_k]

In [74]:
# Iteration 1: Using small model
results=semantic_search(query,documents,embeddings_sm)
results

[(np.float64(0.6756163283889609),
  'LangChain is a framework for developing applications powered by language models'),
 (np.float64(0.1302407042251165),
  'Python is a high-level programming language'),
 (np.float64(0.10111222464316443),
  'Embeddings convert text into numerical vectors')]

In [75]:
# Iteration 2: Using small model
results=semantic_search(query,documents,embeddings_lg)
results

[(np.float64(0.6303544079983328),
  'LangChain is a framework for developing applications powered by language models'),
 (np.float64(0.1272861489017166),
  'Embeddings convert text into numerical vectors'),
 (np.float64(0.12028510264170311),
  'Machine learning is a subset of artificial intelligence')]

## RAG

In [102]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

# Load documents from directory
loader = DirectoryLoader(
    "data/story", 
    glob="*.txt", 
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)
documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:100] + "...")


Loaded 1 documents

First document preview:
Bella met Emma during a late-night coding sprint, both chasing a 
stubborn bug that refused to die. ...


In [103]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]  # Hierarchy of separators
)
chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 2 chunks from 1 documents

Chunk example:
Content: Bella met Emma during a late-night coding sprint, both chasing a 
stubborn bug that refused to die. Frustration turned into laughter, and laughter int...
Metadata: {'source': 'data\\story\\plot.txt'}


In [104]:
chunks

[Document(metadata={'source': 'data\\story\\plot.txt'}, page_content='Bella met Emma during a late-night coding sprint, both chasing a \nstubborn bug that refused to die. Frustration turned into laughter, and laughter into friendship. \nThey built apps, broke systems, and rebuilt each other through failures and breakthroughs. Years passed. Startups rose and fell. Technologies changed. Their bond didn’t. Forty years later, Bella sat beside Emma, now frail but smiling, recalling their first commit together. Emma whispered, “We debugged life pretty well.” When she was'),
 Document(metadata={'source': 'data\\story\\plot.txt'}, page_content='“We debugged life pretty well.” When she was gone, Bella opened their old repository, ran the code, \nand smiled through tears. Some programs end, but some connections keep running forever.')]

In [105]:
from langchain_community.vectorstores import Chroma

In [106]:
## Create a Chromdb vector store
persist_directory="./chroma_db"

## Initialize Chromadb with Open AI embeddings
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_sm,
    persist_directory=persist_directory,
    collection_name="plot_collection"

)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

Vector store created with 2 vectors
Persisted to: ./chroma_db


In [108]:
query="Who is Bella?"

similar_docs=vectorstore.similarity_search(query, k=2)
similar_docs

[Document(metadata={'source': 'data\\story\\plot.txt'}, page_content='Bella met Emma during a late-night coding sprint, both chasing a \nstubborn bug that refused to die. Frustration turned into laughter, and laughter into friendship. \nThey built apps, broke systems, and rebuilt each other through failures and breakthroughs. Years passed. Startups rose and fell. Technologies changed. Their bond didn’t. Forty years later, Bella sat beside Emma, now frail but smiling, recalling their first commit together. Emma whispered, “We debugged life pretty well.” When she was'),
 Document(metadata={'source': 'data\\story\\plot.txt'}, page_content='“We debugged life pretty well.” When she was gone, Bella opened their old repository, ran the code, \nand smiled through tears. Some programs end, but some connections keep running forever.')]

In [88]:
query="what is python?"

similar_docs=vectorstore.similarity_search(query, k=2)
similar_docs

[Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.'),
 Document(metadata={'source': 'data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised 

In [87]:
results_scores=vectorstore.similarity_search_with_score(query, k=2)
results_scores

[(Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.'),
  0.7867954969406128),
 (Document(metadata={'source': 'data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine 

In [90]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name="gpt-4o-mini-2024-07-18",
    api_key = api_key
)

In [125]:
test_response=llm.invoke("Who is Bella and Emma .Respond within 20 words. If you don't know say I don't know without hallucinating")
print(test_response)

AIMessage(
    content="I don't know.",
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 4,
            'prompt_tokens': 30,
            'total_tokens': 34,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-4o-mini-2024-07-18',
        'system_fingerprint': 'fp_6a5cd723c4',
        'id': 'chatcmpl-DTOnQ4X4G2C9VwXncTulU2VGl9VPz',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019d7bd7-47b9-7f12-8235-df546219e910-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 30,
        'output_tokens': 4,
        'total_tokens': 34,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)

In [112]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
    search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
)
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000022AF8F67EF0>, search_kwargs={})

In [113]:
## Create a prompt template
from langchain_core.prompts import ChatPromptTemplate
system_prompt="""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [115]:
### Create a document chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain=create_stuff_documents_chain(llm, prompt)
document_chain


RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000022AF8E75670>, async_client=<openai.res

This chain:

- Takes retrieved documents
- "Stuffs" them into the prompt's {context} placeholder
- Sends the complete prompt to the LLM
- Returns the LLM's response

#### What is create_retrieval_chain?
create_retrieval_chain is a function that combines a retriever (which fetches relevant documents) with a document chain (which processes those documents with an LLM) to create a complete RAG pipeline.

In [116]:
### Create The Final RAG Chain
from langchain_classic.chains import create_retrieval_chain
rag_chain=create_retrieval_chain(retriever, document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000022AF8F67EF0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don'

In [119]:
response=rag_chain.invoke({"input":"Who is Bella and Emma .Respond within 20 words. If you don't know say I don't know without hallucinating "})

In [120]:
response

{'input': "Who is Bella and Emma .Respond within 20 words. If you don't know say I don't know without hallucinating ",
 'context': [Document(metadata={'source': 'data\\story\\plot.txt'}, page_content='Bella met Emma during a late-night coding sprint, both chasing a \nstubborn bug that refused to die. Frustration turned into laughter, and laughter into friendship. \nThey built apps, broke systems, and rebuilt each other through failures and breakthroughs. Years passed. Startups rose and fell. Technologies changed. Their bond didn’t. Forty years later, Bella sat beside Emma, now frail but smiling, recalling their first commit together. Emma whispered, “We debugged life pretty well.” When she was'),
  Document(metadata={'source': 'data\\story\\plot.txt'}, page_content='“We debugged life pretty well.” When she was gone, Bella opened their old repository, ran the code, \nand smiled through tears. Some programs end, but some connections keep running forever.')],
 'answer': 'Bella and Emma ar

In [123]:
from rich import print
print(response)

{
    'input': "Who is Bella and Emma .Respond within 20 words. If you don't know say I don't know without 
hallucinating ",
    'context': [
        Document(
            metadata={'source': 'data\\story\\plot.txt'},
            page_content='Bella met Emma during a late-night coding sprint, both chasing a \nstubborn bug that 
refused to die. Frustration turned into laughter, and laughter into friendship. \nThey built apps, broke systems, 
and rebuilt each other through failures and breakthroughs. Years passed. Startups rose and fell. Technologies 
changed. Their bond didn’t. Forty years later, Bella sat beside Emma, now frail but smiling, recalling their first 
commit together. Emma whispered, “We debugged life pretty well.” When she was'
        ),
        Document(
            metadata={'source': 'data\\story\\plot.txt'},
            page_content='“We debugged life pretty well.” When she was gone, Bella opened their old repository, ran
the code, \nand smiled through tears. Some programs end, but some connections keep running forever.'
        )
    ],
    'answer': 'Bella and Emma are friends who bonded over coding and debugging, sharing a lifelong connection 
through their experiences.'
}

### LCEL

In [127]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [128]:
# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [126]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [129]:
## Build the chain ussing LCEL

rag_chain_lcel=(
    { 
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000022AF8F67EF0>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000022AF8E75670>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000022AF662A270>, r

In [130]:
response=rag_chain_lcel.invoke("Who are Bella and Emma .Respond within 20 words. If you don't know say I don't know without hallucinating")
print(response)

Bella and Emma are friends and coders who shared a lifelong bond through their experiences in software development.

In [131]:
# Add new documents to the existing vector store
new_document = """
Alice joined Bella and Emma during their early coding days,
 bringing vision beyond code. 
 While they solved technical challenges, she focused on building impactful products. Her leadership and clarity set direction. 
 Over time, Alice founded a company, became a successful CEO, and carried their shared passion into the world.
"""

In [132]:
new_doc=Document(
    page_content=new_document,
    metadata={"source": "manual_addition", "topic": "alice_introduction"}
)

In [133]:
## split the documents
new_chunks=text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'alice_introduction'}, page_content='Alice joined Bella and Emma during their early coding days,\n bringing vision beyond code. \n While they solved technical challenges, she focused on building impactful products. Her leadership and clarity set direction. \n Over time, Alice founded a company, became a successful CEO, and carried their shared passion into the world.')]

In [134]:
response=rag_chain_lcel.invoke("Do you know who Alice is in that story? .Respond within 20 words. If you don't know say I don't know without hallucinating")
print(response)

I don't know.

In [135]:
vectorstore.add_documents(new_chunks)

['f8bb0fa5-5986-4b63-a252-beec6b67e34e']

In [136]:
response=rag_chain_lcel.invoke("Do you know who Alice is in that story? .Respond within 20 words. If you don't know say I don't know without hallucinating")
print(response)

Alice is a leader who brought vision to coding, founded a company, and became a successful CEO.